<a href="https://colab.research.google.com/github/harshs-data/Pytorch/blob/main/Optimizing_the_Neural_Network.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt


In [20]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("zalando-research/fashionmnist")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'fashionmnist' dataset.
Path to dataset files: /kaggle/input/fashionmnist


In [21]:
import pandas as pd

# Assuming there is a file named 'fashion-mnist_train.csv' in that folder
# Adjust the filename based on what you saw in Step 2
df = pd.read_csv(f"{path}/fashion-mnist_train.csv")

# Look at the first 5 rows
df

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59995,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
59996,1,0,0,0,0,0,0,0,0,0,...,73,0,0,0,0,0,0,0,0,0
59997,8,0,0,0,0,0,0,0,0,0,...,160,162,163,135,94,0,0,0,0,0
59998,8,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [22]:
# check for GPU

import torch

# This automatically picks GPU if available, otherwise stays on CPU
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Using device: {device}")

Using device: cuda


In [23]:
X = df.iloc[:,1:].values
y = df.iloc[:,0].values

In [24]:
# train test split
X_train, X_test, y_train, y_test =train_test_split(X,y, test_size=0.20, random_state=42)

In [25]:
X_train = X_train/255.0
X_test = X_test/255.0

In [26]:
class CustomDataset(Dataset):

  def __init__(self, features, labels):

    self.features = torch.tensor(features, dtype = torch.float32)
    self.labels = torch.tensor(labels, dtype=torch.long)

  def __len__(self):
    return len(self.features)

  def __getitem__(self, idx):
    return self.features[idx], self.labels[idx]



In [27]:
train_dataset = CustomDataset(X_train, y_train)
test_dataset = CustomDataset(X_test, y_test)

In [34]:
len(train_dataloader) # no. of batches created

1500

In [29]:
class MyNN(nn.Module):

  def __init__(self, input_dim, output_dim, num_hidden_layers, neurons_per_layer, dropout_rate):

    super().__init__()

    layers = []

    for i in range(num_hidden_layers):

      layers.append(nn.Linear(input_dim, neurons_per_layer))
      layers.append(nn.BatchNorm1d(neurons_per_layer))
      layers.append(nn.ReLU())
      layers.append(nn.Dropout(dropout_rate))
      input_dim = neurons_per_layer

    layers.append(nn.BatchNorm1d(neurons_per_layer, output_dim))

    self.model = nn.Sequential(*layers)

  def forward(self, X):
    return self.model(X)

In [35]:
# objective function

def objective(trial):

  # hyperparameter values with search space
  num_hidden_layers = trial.suggest_int("num_hidden_layers", 1,5)
  neurons_per_layer = trial.suggest_int('neurons_per_layer', 8,128, step=8)
  epochs = trial.suggest_int("epochs", 10,50, step=10)
  learning_rate = trial.suggest_float("learning_rate",1e-5,1e-1, log=True)
  dropout_rate = trial.suggest_float("dropout_rate", 0.1, 0.5, step=0.1)
  batch_size = trial.suggest_categorical("batch_size", [16,32,64,128])
  optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "SGD", "RMSprop"])
  Weight_decay = trial.suggest_float("Weight_decay", 1e-5, 1e-1, log=True)


  # dataloader

  train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
  test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

  # model init
  input_dim = 784
  output_dim = 10

  model = MyNN(input_dim, output_dim, num_hidden_layers, neurons_per_layer, dropout_rate)
  model = model.to(device)



  # loss function
  criterian = nn.CrossEntropyLoss()
  # Initialize optimizer with a default, then update based on choice
  optimizer = None

  if optimizer_name == "Adam":
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=Weight_decay)

  elif optimizer_name == "RMSprop":
    optimizer = torch.optim.RMSprop(model.parameters(), lr=learning_rate, weight_decay=Weight_decay)

  else:
    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, weight_decay=Weight_decay)


  # training loop
  for epoch in range(epochs):

    for batch_features, batch_labels in train_dataloader:

      # Move data to GPU

      batch_features = batch_features.to(device)
      batch_labels = batch_labels.to(device)

      # forward pass
      outputs = model(batch_features)

      # loss
      loss = criterian(outputs, batch_labels)

      # back pass
      optimizer.zero_grad()
      loss.backward()

      # update weights
      optimizer.step()


  # evaluation
  model.eval()

  # evaluation code

  total = 0
  correct = 0

  with torch.no_grad():

    for batch_features, batch_labels in test_dataloader:

      # move to GPU
      batch_features = batch_features.to(device)
      batch_labels = batch_labels.to(device)

      outputs = model(batch_features)

      _,predicted = torch.max(outputs,1)

      total = total + batch_labels.size(0)

      correct = correct + (predicted == batch_labels).sum().item()


    accuracy = correct/total



  return accuracy

In [14]:
! pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 33.5 MB/s eta 0:00:00


In [31]:
import optuna

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10)

[I 2026-04-19 13:50:08,025] A new study created in memory with name: no-name-36974e82-1c88-4852-9bea-1382682f655c
[I 2026-04-19 13:51:04,721] Trial 0 finished with value: 0.6979166666666666 and parameters: {'num_hidden_layers': 2, 'neurons_per_layer': 16, 'epochs': 50, 'learning_rate': 0.02835235658892544, 'dropout_rate': 0.5, 'batch_size': 128, 'optimizer': 'Adam', 'Weight_decay': 0.00047088981013272306}. Best is trial 0 with value: 0.6979166666666666.
[I 2026-04-19 13:54:42,735] Trial 1 finished with value: 0.8625 and parameters: {'num_hidden_layers': 5, 'neurons_per_layer': 80, 'epochs': 20, 'learning_rate': 1.58879906387244e-05, 'dropout_rate': 0.2, 'batch_size': 16, 'optimizer': 'Adam', 'Weight_decay': 0.0010439339719149315}. Best is trial 1 with value: 0.8625.
[I 2026-04-19 13:58:37,090] Trial 2 finished with value: 0.8426666666666667 and parameters: {'num_hidden_layers': 3, 'neurons_per_layer': 64, 'epochs': 30, 'learning_rate': 1.4282224472528188e-05, 'dropout_rate': 0.4, 'batc

In [32]:
study.best_value

0.8815833333333334

In [33]:
study.best_params

{'num_hidden_layers': 2,
 'neurons_per_layer': 64,
 'epochs': 30,
 'learning_rate': 0.08371591838009115,
 'dropout_rate': 0.2,
 'batch_size': 32,
 'optimizer': 'SGD',
 'Weight_decay': 1.4201109639398865e-05}